# 第一个 LangChain Agent

Agent 是 LangChain 最核心的概念，它能让 AI 自动判断何时需要调用工具，并在获取工具结果后继续思考，直到完成任务。

## 什么是 Agent

普通的模型调用是一问一答：

```
你发消息，模型回消息，结束。
```

Agent 则不同，它进入一个 **思考-行动-观察** 循环：

```
模型判断需要调用某个工具 → 执行工具并拿到结果 → 模型根据结果继续思考 → 可能再调用工具 → 直到得出最终答案。
```

举个例子，如果你问"今天杭州天气怎么样？"，普通模型只能回答训练数据中的天气（可能是几个月前的）。而 Agent 会主动调用天气查询工具获取实时数据，然后基于真实数据回答你。

**术语：**

- **Agent**：能自主调用工具并循环思考的智能体
- **思考-行动-观察循环**：Agent 的核心执行模式
  - **思考**：模型判断需要做什么
  - **行动**：调用工具执行操作
  - **观察**：分析工具结果，决定下一步

## 创建第一个 Agent

整个过程只需要三步：(1) 定义工具；(2) 创建 Agent；(3) 运行。

### 步骤 1：定义工具函数

使用 `@tool` 装饰器，把普通 Python 函数变成 Agent 可以调用的工具：

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.tools import tool

# 步骤 1：用 @tool 装饰器定义一个工具
# 函数的文档字符串就是工具的描述
# 模型会根据描述来判断何时调用这个工具
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气情况。
    
    Args:
        city: 城市名称，如 "杭州"、"北京"
    """
    # 这里用模拟数据演示
    # 实际项目中可以替换为真实的天气 API 调用
    weather_data = {
        "杭州": "晴，25°C，湿度 60%",
        "北京": "多云，18°C，湿度 45%",
        "上海": "小雨，22°C，湿度 80%",
    }
    return weather_data.get(city, f"未找到 {city} 的天气数据")

@tool
def calculate(expression: str) -> str:
    """执行数学计算。支持加减乘除等基本运算。
    
    Args:
        expression: 数学表达式，如 "3 * 7 + 2"
    """
    try:
        # 安全地计算数学表达式
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"

print("工具定义完成！")

工具定义完成！


> 文档字符串（函数的 `"""..."""` 部分）非常重要。模型会读取工具的描述来决定是否调用这个工具以及传什么参数。描述越清晰，模型就越不容易用错。

**术语：**

- **@tool 装饰器**：将 Python 函数标记为 Agent 可调用的工具
- **工具描述（Tool Description）**：函数的文档字符串，告诉模型这个工具的作用

### 步骤 2：创建 Agent

In [2]:
# 步骤 2：创建 Agent
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 初始化模型（使用项目配置的 DeepSeek）
model = init_chat_model("deepseek:deepseek-v4-flash")

# 创建 Agent，传入模型和工具列表
agent = create_agent(
    model=model,
    tools=[get_weather, calculate],
    system_prompt="你是一个乐于助人的助手，会使用工具来回答问题。",
)

print("Agent 创建完成！")

Agent 创建完成！


| 参数 | 说明 | 是否必填 |
| --- | --- | --- |
| model | 使用的语言模型 | 是 |
| tools | 工具列表，Agent 可以调用这些工具 | 否（不传则 Agent 无法调用工具） |
| system_prompt | 系统提示词，定义 Agent 的角色和行为 | 否 |

**术语：**

- **create_agent()**：LangChain 的高阶函数，用于创建 Agent
- **system_prompt**：系统提示词，定义 Agent 的角色和行为规则
- **init_chat_model()**：统一模型初始化函数

### 步骤 3：运行 Agent

In [3]:
# 步骤 3：运行 Agent

# 构建输入消息
# 消息列表中的第一条通常是 HumanMessage（用户消息）
from langchain.messages import HumanMessage

inputs = {"messages": [HumanMessage(content="杭州今天天气怎么样？")]}

# invoke() 运行 Agent，返回最终状态
result = agent.invoke(inputs)

# 查看消息历史（包含 AI 的工具调用和工具返回结果）
print("=== 完整消息历史 ===")
for msg in result["messages"]:
    print(f"[{msg.type}] {msg.content[:100]}")  # 截取前 100 字符

print("\n=== 最终回复 ===")
# 最后一条 AI 消息就是最终答案
print(result["messages"][-1].content)

=== 完整消息历史 ===
[human] 杭州今天天气怎么样？
[ai] 好的，我来帮你查一下杭州今天的天气情况！
[tool] 晴，25°C，湿度 60%
[ai] 杭州今天天气不错哦！☀️

- **天气状况**：晴
- **气温**：25°C
- **湿度**：60%

总的来说是个阳光明媚的好天气，温度也非常舒适，很适合外出活动。不过湿度适中，建议适当补水哦

=== 最终回复 ===
杭州今天天气不错哦！☀️

- **天气状况**：晴
- **气温**：25°C
- **湿度**：60%

总的来说是个阳光明媚的好天气，温度也非常舒适，很适合外出活动。不过湿度适中，建议适当补水哦！😊


运行结果示例：

```
=== 完整消息历史 ===
[human] 杭州今天天气怎么样？
[ai]
[tool] 晴，25°C，湿度 60%
[ai] 今天杭州的天气是晴天，气温25°C，湿度60%。非常适合出门活动！

=== 最终回复 ===
今天杭州的天气是晴天，气温25°C，湿度60%。非常适合出门活动！
```

## Agent 执行流程解析

上面这个例子的完整执行过程如下：

1. 用户发送消息："杭州今天天气怎么样？"
2. 模型收到消息后判断：需要查询天气 → 返回一个 tool_call（调用 get_weather，参数 city="杭州"）
3. Agent 执行 get_weather 工具 → 返回 "晴，25°C，湿度 60%"
4. 模型收到工具结果 → 判断任务完成 → 生成最终回复

这就是 Agent 的 **思考-行动-观察** 循环。

![](https://www.runoob.com/wp-content/uploads/2026/05/6fde20ee-22ee-4ddf-9824-0d4c667f2e99.png)

## 让 Agent 调用多个工具

Agent 的真正威力在于能自动组合多个工具：

In [4]:
# 问一个需要同时查询天气和计算的复杂问题
inputs = {"messages": [HumanMessage(
    content="杭州和北京今天温差多少度？"
)]}

result = agent.invoke(inputs)

print("=== 完整消息历史 ===")
for msg in result["messages"]:
    if msg.type == "tool":
        print(f"[tool {msg.name}] {msg.content}")
    else:
        print(f"[{msg.type}] {msg.content[:120]}")

print("\n=== 最终回复 ===")
print(result["messages"][-1].content)

=== 完整消息历史 ===
[human] 杭州和北京今天温差多少度？
[ai] 好的，我先查询杭州和北京今天的天气情况！
[tool get_weather] 晴，25°C，湿度 60%
[tool get_weather] 多云，18°C，湿度 45%
[ai] 好的，我来算一下温差。
[tool calculate] 计算结果: 25 - 18 = 7
[ai] 查询到结果啦！以下是今天（当前）的天气情况：

- **杭州** ☀️：晴，**25°C**，湿度 60%
- **北京** ⛅：多云，**18°C**，湿度 45%

**温差：** 杭州比北京高 **7°C** 🌡️（25°C - 18

=== 最终回复 ===
查询到结果啦！以下是今天（当前）的天气情况：

- **杭州** ☀️：晴，**25°C**，湿度 60%
- **北京** ⛅：多云，**18°C**，湿度 45%

**温差：** 杭州比北京高 **7°C** 🌡️（25°C - 18°C = 7°C）

北京今天比杭州凉快不少，如果在两地之间出行的话，记得适当增减衣物哦！


运行结果示例：

```
=== 完整消息历史 ===
[human] 杭州和北京今天温差多少度？
[ai]
[tool get_weather] 晴，25°C，湿度 60%
[tool get_weather] 多云，18°C，湿度 45%
[tool calculate] 计算结果: 25 - 18 = 7
[ai] 今天杭州和北京的温差是7°C。

=== 最终回复 ===
今天杭州和北京的温差是7°C。
```

Agent 自动执行了三步：(1) 查询杭州天气；(2) 查询北京天气；(3) 计算温差。你不需要写任何逻辑来控制这些步骤，Agent 自动完成了规划和执行。

> 注意 Agent 在第一次调用模型时就发出了两个工具调用（get_weather("杭州") 和 get_weather("北京")），它们是并行执行的。模型会自动判断哪些工具调用可以并行。

**术语：**

- **并行工具调用**：Agent 可以同时调用多个独立的工具，提高效率
- **工具规划（Tool Planning）**：模型自动决定调用哪些工具、按什么顺序调用

## 完整代码

整合以上代码，形成一个完整的示例：

In [5]:
# 完整示例
from dotenv import load_dotenv
load_dotenv()

from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

# 定义工具
@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气情况。
    
    Args:
        city: 城市名称，如 "杭州"、"北京"
    """
    weather_data = {
        "杭州": "晴，25°C，湿度 60%",
        "北京": "多云，18°C，湿度 45%",
        "上海": "小雨，22°C，湿度 80%",
    }
    return weather_data.get(city, f"未找到 {city} 的天气数据")

@tool
def calculate(expression: str) -> str:
    """执行数学计算。支持加减乘除等基本运算。
    
    Args:
        expression: 数学表达式，如 "3 * 7 + 2"
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"

# 创建 Agent
model = init_chat_model("deepseek:deepseek-v4-flash")
agent = create_agent(
    model=model,
    tools=[get_weather, calculate],
    system_prompt="你是一个乐于助人的助手，会使用工具来回答问题。",
)

# 运行 Agent
def ask(question: str):
    """发送问题到 Agent 并打印结果"""
    inputs = {"messages": [HumanMessage(content=question)]}
    result = agent.invoke(inputs)
    print(f"问题: {question}")
    print(f"回答: {result['messages'][-1].content}")
    print("-" * 50)
    return result

# 测试几个问题
ask("杭州今天天气怎么样？")
ask("杭州和北京今天温差多少度？")
ask("菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？")

问题: 杭州今天天气怎么样？
回答: 杭州今天天气情况如下：

- **天气**：☀️ 晴天
- **温度**：🌡️ 25°C
- **湿度**：💧 60%

今天杭州天气不错，阳光明媚，温度也比较舒适，非常适合出门活动哦！不过湿度稍微有点高，记得多喝水~ 😊
--------------------------------------------------
问题: 杭州和北京今天温差多少度？
回答: 今天杭州和北京的温差情况如下：

- **杭州**：☀️ 晴，**25°C**
- **北京**：⛅ 多云，**18°C**

两地温差为 **7°C**，杭州比北京暖和一些。如果你有出行计划，记得根据气温适当增减衣物哦！
--------------------------------------------------
问题: 菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？
回答: 3 个朋友推荐，再加上 2 个，一共是 **5 个人推荐**！🎉

菜鸟教程 RUNOOB 得到这么多人的推荐，说明它确实是一个学习编程和技术知识的好去处呀！😄
--------------------------------------------------


{'messages': [HumanMessage(content='菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？', additional_kwargs={}, response_metadata={}, id='de91258c-c923-45b8-9472-62d0b4425537'),
  AIMessage(content='菜鸟教程 RUNOOB 确实是一个非常棒的学习平台！🎉\n\n至于你的问题，我们来算一下：', additional_kwargs={'tool_calls': [{'id': 'call_00_6Fo9SqRhdDZUbXsu3Py30374', 'function': {'arguments': '{"expression": "3 + 2"}', 'name': 'calculate'}, 'type': 'function', 'index': 0}], 'refusal': None, 'reasoning_content': '用户的问题有两个部分：\n1. 提及了菜鸟教程 RUNOOB 这个学习平台\n2. 问了一个数学问题：3个朋友推荐，再加上2个，一共多少人推荐？\n\n这其实是一个简单的数学计算：3 + 2 = 5。\n\n让我用计算工具来计算一下。'}, response_metadata={'token_usage': {'completion_tokens': 132, 'prompt_tokens': 408, 'total_tokens': 540, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 64, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 152}, 'model_name':

运行结果：

```
问题: 杭州今天天气怎么样？
回答: 今天杭州的天气是晴天，气温25°C，湿度60%。非常适合出门活动！
--------------------------------------------------
问题: 杭州和北京今天温差多少度？
回答: 今天杭州和北京的温差是7°C。
--------------------------------------------------
问题: 菜鸟教程 RUNOOB 是一个非常棒的学习平台，如果我有 3 个朋友都推荐了，再加上 2 个，一共多少人推荐？
回答: 一共 5 人推荐了菜鸟教程 RUNOOB！
--------------------------------------------------
```

> 注意第三个问题：Agent 能够理解"推荐人数"是一个计算问题，并自动调用 calculate 工具。这说明工具调用的决策是由模型对语义的理解来驱动的，而不是硬编码的规则。

## 同步 vs 异步运行

Agent 支持异步模式，适合在 Web 服务等异步环境中使用：

In [7]:
# 异步运行 Agent
# Notebook 环境中可以直接使用 await，无需 asyncio.run()
import asyncio
from langchain.messages import HumanMessage

# 直接 await 异步方法（IPython 7+ 支持顶层 await）
inputs = {"messages": [HumanMessage(content="杭州天气怎么样？")]}
result = await agent.ainvoke(inputs)
print(result["messages"][-1].content)


杭州现在的天气情况如下：

🌤️ **天气：** 晴  
🌡️ **温度：** 25°C  
💧 **湿度：** 60%  

天气不错，阳光明媚，温度也很舒适，适合出门活动哦！不过湿度稍高，体感可能会有点闷。有什么其他需要帮忙的吗？😊


**术语：**

- **invoke()**：同步运行 Agent
- **ainvoke()**：异步运行 Agent（适合 Web 服务器等异步环境）
- **asyncio**：Python 异步编程库

## 小结

到这里你已经掌握了 LangChain 最核心的用法：

- 用 `@tool` 装饰器把 Python 函数变成工具
- 用 `create_agent()` 创建能自动调用工具的 Agent
- 用 `agent.invoke()` 运行 Agent 并获得结果

**核心概念回顾：**

| 概念 | 说明 |
| --- | --- |
| **Agent** | 能自主调用工具并循环思考的智能体 |
| **工具（Tool）** | Agent 可以调用的 Python 函数 |
| **思考-行动-观察** | Agent 的核心执行模式 |
| **create_agent()** | 创建 Agent 的核心函数 |
| **invoke() / ainvoke()** | 运行 Agent 的方法 |

## 接下来的学习路径

本教程接下来的 Notebook 将深入这些核心概念：

| # | Notebook | 内容 |
| --- | --- | --- |
| 01 | 模型调用 | init_chat_model() 函数详解 |
| 02 | 常用参数 | temperature、max_tokens 等参数说明 |
| 03 | 高级用法 | 流式输出、结构化输出等 |
| 04 | 消息类型 | System/Human/AI 消息详解 |
| 05 | @tool 基础 | 工具定义的更多细节 |
| 06 | 工具高级特性 | 错误处理、工具组合等 |
| 07 | 状态存储 | InjectedState 与 InjectedStore |
| 08 | create_agent() | 函数参数详解 |
| 09 | Agent 工作流程 | 深入理解 Agent 执行流程 |
| 10 | AgentState | 状态管理机制 |
| 11 | 系统提示词 | System Prompt 与 Dynamic Prompt |
| 12 | 流式输出 | Streaming 详解 |